# Deploy — Match Rate (Track A orchestrator)

> **[Unverified — requires Fabric tenant pilot + F2 capacity]**  ·  **Label: [NET-NEW]**
> Chains the **flagship** query→identity attribution flow end to end:
> land gateway logs → confirm the identity join → **measure the match rate**.
> This is **Track A** of `docs/RUNBOOK-F2-pilot.md`.
>
> **Prereq:** Workspace Monitoring enabled on an **F2+** capacity (trial can't provision the
> Eventhouse — see `LIVE-TENANT-FINDINGS.md`), and a gateway-routed refresh has occurred.
>
> The KQL (`01_identity_join`, `PILOT-identity-join-test`, `04_identity_match_rate`) runs
> against the Monitoring KQL database — run those blocks in a **KQL Queryset**, not here.
> This notebook orchestrates the **log-landing** step and points you to each KQL block.

In [ ]:
# =============================================================================
# (a) CONFIG + CHAIN HELPER  |  Label: [NET-NEW]
# [Unverified — not executed in live Fabric]
# =============================================================================
import os, json, importlib.util

CONFIG_PATH = os.getenv("CONFIG_PATH", "../config/config.json")
NB_DIR = os.getenv("NB_DIR", "../notebooks")

def load_config():
    if os.path.exists(CONFIG_PATH):
        cfg = json.load(open(CONFIG_PATH))
    else:
        print(f"[warn] {CONFIG_PATH} not found — running with empty config (MOCK only).")
        cfg = {}
    # Resolve SP secret from Key Vault at runtime (never store the secret in config.json).
    kv = cfg.get("keyVault") or {}
    if kv.get("vaultUri") and kv.get("spClientSecretSecretName") and not os.getenv("TENANT_EXTRACT_MOCK"):
        try:
            from azure.identity import DefaultAzureCredential
            from azure.keyvault.secrets import SecretClient
            sc = SecretClient(vault_url=kv["vaultUri"], credential=DefaultAzureCredential())
            cfg["clientSecret"] = sc.get_secret(kv["spClientSecretSecretName"]).value
        except Exception as e:  # noqa: BLE001
            print(f"[warn] Key Vault secret fetch failed ({e}); set clientSecret another way for live runs.")
    return cfg

def run_notebook(name):
    """Import + return a sibling medallion notebook module (00_tenant_extract, etc.).
    In Fabric prefer:  %run {name}   (uncomment the magic below). Off-cluster we importlib."""
    # In Fabric, replace the importlib block with:  %run ../notebooks/{name}
    path = os.path.join(NB_DIR, f"{name}.py")
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

CONFIG = load_config()
print("[config] loaded keys:", sorted(k for k in CONFIG if not k.startswith("_")))


## (b) Land gateway logs → bronze — `01_bronze_ingest`
Routes through `gateway_bronze_lib.read_gateway_csv` (the `(ms)`/`(bytes)`-safe path).
**Do NOT** use Fabric Load-to-Tables on the raw CSV (it fails at schema inference).

In [ ]:
# =============================================================================
# (b) LAND GATEWAY LOGS — 01_bronze_ingest  |  Label: [NET-NEW]
# [Unverified — needs the gateway QueryExecution/QueryStart CSVs staged in LANDING_PATH]
# =============================================================================
# In Fabric:  %run ../notebooks/01_bronze_ingest
# Off-cluster the module runs its ingest on import via the __main__-style guard;
# here we import and call the gateway-log entrypoint explicitly.
ingest = run_notebook("01_bronze_ingest")
try:
    ingest.ingest_gateway_logs()   # primary path -> bronze_query_execution / bronze_query_start
    print("[ingest] gateway logs landed to bronze.")
except Exception as e:  # noqa: BLE001
    print(f"[ingest] entrypoint not runnable off-cluster / no CSVs staged: {e}")
    print("        On the pilot host, ensure QueryExecutionReport*.csv is in LANDING_PATH.")


## (c) Confirm the join returns rows — KQL
Open a **KQL Queryset** on the Monitoring KQL database and run, in order:
1. `starter/kql/PILOT-identity-join-test.kql` Block 1 — recent monitoring rows exist.
2. Block 2 — paste a `RequestId` from the gateway host; confirm `ExecutingUser` + `ItemName`.
3. Block 3 — `getschema` to confirm live column names (report any drift).

In [ ]:
# =============================================================================
# (c) POINTERS — the KQL to run in a Queryset (not executable here)
# =============================================================================
print("Run in a KQL Queryset on the Monitoring KQL DB:")
print("  1) starter/kql/PILOT-identity-join-test.kql   (Blocks 1-3: confirm the join)")
print("  2) starter/kql/01_identity_join.kql           (the join itself)")
print("  3) starter/kql/04_identity_match_rate.kql     (Block A = match_rate_pct)")


## (d) Measure the match rate — `04_identity_match_rate.kql`
In the KQL Queryset run **Block A** for the headline `match_rate_pct`, **Block B** for the
Refresh-vs-DirectQuery split, **Block C** to sample unattributed queries if the rate is low.
Keep `lookback = 15m` + one workspace GUID on a throttled capacity.

## (e) Report back — Track A

File a pilot-report issue with the Block A counts + `match_rate_pct`, the Block B split,
the capacity SKU (F2), and any schema drift. See **Track A** in
[`docs/RUNBOOK-F2-pilot.md`](../../docs/RUNBOOK-F2-pilot.md). Then **pause the F2 capacity**.